# OntodockerClient Demo

`OntodockerClient` is courier's service-specific client for Ontodocker dataset, endpoint, and SPARQL workflows.

This notebook assumes Jupyter was started from a shell where `secrets/.env` was sourced.

## Environment

Add these entries to `secrets/.env`:

```bash
export ONTODOCKER_ADDRESS="https://..."
export ONTODOCKER_TOKEN="..."
```

`ONTODOCKER_TOKEN` may be empty if the service does not need authentication.

In [1]:
import os
from pathlib import Path

import rdflib


def require_env(name: str) -> str:
    value = os.getenv(name)
    if value is None or not value.strip():
        raise RuntimeError(f"Required environment variable {name} is not set.")
    return value.strip()


address = require_env("ONTODOCKER_ADDRESS")
token = os.getenv("ONTODOCKER_TOKEN") or None

In [2]:
from courier import OntodockerClient

client = OntodockerClient(address=address, token=token)
client.base_url

'https://ontodocker.pmds-mpcdf-2.pyiron.com'

## Resource Layout

The client exposes small resource objects for the main service areas.

In [3]:
client.endpoints, client.datasets, client.sparql

(EndpointsResource(client=<courier.services.ontodocker.client.OntodockerClient object at 0x7e140350ba10>, rectify_legacy=True),
 DatasetsResource(client=<courier.services.ontodocker.client.OntodockerClient object at 0x7e140350ba10>),
 SparqlResource(client=<courier.services.ontodocker.client.OntodockerClient object at 0x7e140350ba10>))

## Discover Endpoints And Datasets

Endpoint discovery returns raw endpoint URLs or parsed `EndpointInfo` objects. Dataset listing is derived from endpoint discovery.

In [4]:
raw_endpoints = client.endpoints.list_raw()
raw_endpoints

['https://ontodocker.pmds-mpcdf-2.pyiron.com/api/v1/jena/pmdco2_tto_example/sparql']

In [5]:
endpoints = client.endpoints.list()
endpoints

[EndpointInfo(dataset='pmdco2_tto_example', sparql_endpoint='https://ontodocker.pmds-mpcdf-2.pyiron.com/api/v1/jena/pmdco2_tto_example/sparql')]

In [6]:
datasets = client.datasets.list()
datasets

['pmdco2_tto_example']

## Create A Disposable Dataset

This acts directly on the Ontodocker service. Use a dataset name that is safe to create and delete.

In [7]:
dataset = "courier_demo"

In [8]:
client.datasets.create(dataset)

'"Dataset name courier_demo created"'

## Uploading A Turtle File To An Existing Dataset

In [9]:
turtle_path = Path("resources/test_dataset.ttl")
assert turtle_path.exists(), turtle_path
turtle_path

PosixPath('resources/test_dataset.ttl')

In [10]:
client.datasets.upload_turtlefile(dataset, str(turtle_path))

'"Upload succeeded { \\n  \\"count\\" : 13 ,\\n  \\"tripleCount\\" : 13 ,\\n  \\"quadCount\\" : 0\\n}\\n"'

## Upload A `rdflib.Graph`

`rdflib.Graph.parse()` reads the same Turtle file used for the file-upload example. `upload_graph()` then serializes that graph to Turtle in memory and uploads it as multipart form data.

In [11]:
graph = rdflib.Graph()
graph.parse(turtle_path, format="turtle")

<Graph identifier=N3970a0426f1f400096a6b06a380dffc4 (<class 'rdflib.graph.Graph'>)>

In [12]:
client.datasets.upload_graph(dataset, graph)

'"Upload succeeded { \\n  \\"count\\" : 13 ,\\n  \\"tripleCount\\" : 13 ,\\n  \\"quadCount\\" : 0\\n}\\n"'

## Fetch Or Download Turtle

In [13]:
client.datasets.fetch_turtle(dataset)

'@prefix ex:   <http://example.org/> .\n@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\n@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .\n\nex:Person  a        rdfs:Class;\n        rdfs:label  "Person"@en .\n\nex:Project  a       rdfs:Class;\n        rdfs:label  "Project"@en .\n\nex:worksOn  a        rdfs:Property;\n        rdfs:domain  ex:Person;\n        rdfs:label   "works on"@en;\n        rdfs:range   ex:Project .\n\nex:alice  a         ex:Person;\n        rdfs:label  "Alice"@en;\n        ex:worksOn  ex:projectX .\n\nex:projectX  a      ex:Project;\n        rdfs:label  "Project X"@en .\n'

In [14]:
client.datasets.download_turtle(dataset, "downloaded_demo.ttl")

PosixPath('downloaded_demo.ttl')

## Making SPARQL Queries

In [15]:
client.sparql.endpoint(dataset)

'https://ontodocker.pmds-mpcdf-2.pyiron.com/api/v1/jena/courier_demo/sparql'

In [16]:
query = """
SELECT ?s ?p ?o
WHERE {{ ?s ?p ?o }}
"""

In [17]:
client.sparql.query_raw(dataset, query)

'{"head":{"vars":["s","p","o"]},"results":{"bindings":[{"s":{"type":"uri","value":"http://example.org/Person"},"p":{"type":"uri","value":"http://www.w3.org/1999/02/22-rdf-syntax-ns#type"},"o":{"type":"uri","value":"http://www.w3.org/2000/01/rdf-schema#Class"}},{"s":{"type":"uri","value":"http://example.org/Person"},"p":{"type":"uri","value":"http://www.w3.org/2000/01/rdf-schema#label"},"o":{"type":"literal","xml:lang":"en","value":"Person"}},{"s":{"type":"uri","value":"http://example.org/Project"},"p":{"type":"uri","value":"http://www.w3.org/1999/02/22-rdf-syntax-ns#type"},"o":{"type":"uri","value":"http://www.w3.org/2000/01/rdf-schema#Class"}},{"s":{"type":"uri","value":"http://example.org/Project"},"p":{"type":"uri","value":"http://www.w3.org/2000/01/rdf-schema#label"},"o":{"type":"literal","xml:lang":"en","value":"Project"}},{"s":{"type":"uri","value":"http://example.org/worksOn"},"p":{"type":"uri","value":"http://www.w3.org/1999/02/22-rdf-syntax-ns#type"},"o":{"type":"uri","value":

In [18]:
df = client.sparql.query_df(dataset, query, columns=["s", "p", "o"])
df

,s,p,o
0,http://example.org/Person,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2000/01/rdf-schema#Class
1,http://example.org/Person,http://www.w3.org/2000/01/rdf-schema#label,Person
2,http://example.org/Project,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2000/01/rdf-schema#Class
3,http://example.org/Project,http://www.w3.org/2000/01/rdf-schema#label,Project
4,http://example.org/worksOn,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2000/01/rdf-schema#Property
5,http://example.org/worksOn,http://www.w3.org/2000/01/rdf-schema#label,works on
6,http://example.org/worksOn,http://www.w3.org/2000/01/rdf-schema#domain,http://example.org/Person
7,http://example.org/worksOn,http://www.w3.org/2000/01/rdf-schema#range,http://example.org/Project
8,http://example.org/alice,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/Person
9,http://example.org/alice,http://www.w3.org/2000/01/rdf-schema#label,Alice


## Cleanup

In [19]:
# Uncomment after the demo if the dataset should be removed.
client.datasets.delete(dataset)
Path("downloaded_demo.ttl").unlink(missing_ok=True)